# PCMCI + CMI GPU Check

This notebook runs the project's existing PCMCI + CMI pipeline and prints enough runtime information to confirm whether the CMI backend is truly using GPU or has fallen back to CPU.

Notes:
- The notebook uses the code in `src/` directly.
- `FAISSCMI` uses FAISS GPU only when FAISS GPU resources are available.
- On Apple Silicon, PyTorch `mps` may exist, but FAISS does not use MPS here; only FAISS GPU/CUDA matters for CMI.


In [1]:
from __future__ import annotations

import logging
import os
import platform
import sys
from pathlib import Path

CANDIDATES = [Path.cwd(), Path.cwd().parent]
ROOT = next((candidate for candidate in CANDIDATES if (candidate / 'src').exists()), None)
if ROOT is None:
    raise RuntimeError(
        'Could not find the project root from the current notebook directory. '
        f'Tried: {CANDIDATES}'
    )

os.chdir(ROOT)
os.environ.setdefault('MPLCONFIGDIR', '/tmp/mpl')
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('MKL_NUM_THREADS', '1')
os.environ.setdefault('VECLIB_MAXIMUM_THREADS', '1')

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)s %(name)s: %(message)s',
    force=True,
)

print('Notebook cwd before root detection:', CANDIDATES[0])
print('Project root:', ROOT)
print('Working directory reset to:', Path.cwd())
print('KMP_DUPLICATE_LIB_OK:', os.environ.get('KMP_DUPLICATE_LIB_OK'))
print('OMP_NUM_THREADS:', os.environ.get('OMP_NUM_THREADS'))
print('Python:', sys.version)
print('Platform:', platform.platform())
print('Machine:', platform.machine())


Notebook cwd before root detection: /Users/arshia/Projects/Papers/FinancialAI/Gaia/notebooks
Project root: /Users/arshia/Projects/Papers/FinancialAI/Gaia
Working directory reset to: /Users/arshia/Projects/Papers/FinancialAI/Gaia
KMP_DUPLICATE_LIB_OK: TRUE
OMP_NUM_THREADS: 1
Python: 3.10.19 (main, Oct 28 2025, 12:01:32) [Clang 20.1.4 ]
Platform: macOS-14.3-arm64-arm-64bit
Machine: arm64


In [2]:
import importlib

import faiss
import numpy as np
import torch

print('torch version:', torch.__version__)
print('faiss version:', getattr(faiss, '__version__', 'n/a'))
print('torch.cuda.is_available():', torch.cuda.is_available())
print('torch.cuda.device_count():', torch.cuda.device_count())
print('has torch.backends.mps:', hasattr(torch.backends, 'mps'))
if hasattr(torch.backends, 'mps'):
    print('torch.backends.mps.is_built():', torch.backends.mps.is_built())
    print('torch.backends.mps.is_available():', torch.backends.mps.is_available())
print('has faiss.StandardGpuResources:', hasattr(faiss, 'StandardGpuResources'))
print('has faiss.index_cpu_to_gpu:', hasattr(faiss, 'index_cpu_to_gpu'))
if hasattr(faiss, 'get_num_gpus'):
    try:
        print('faiss.get_num_gpus():', faiss.get_num_gpus())
    except Exception as exc:
        print('faiss.get_num_gpus() error:', repr(exc))


2026-05-15 19:11:30,667 INFO faiss.loader: Loading faiss.
2026-05-15 19:11:30,690 INFO faiss.loader: Successfully loaded faiss.


torch version: 2.11.0
faiss version: 1.13.2
torch.cuda.is_available(): False
torch.cuda.device_count(): 0
has torch.backends.mps: True
torch.backends.mps.is_built(): True
torch.backends.mps.is_available(): True
has faiss.StandardGpuResources: False
has faiss.index_cpu_to_gpu: False
faiss.get_num_gpus(): 0


In [3]:
from src.config.load import load_config
from src.pcmci.runner import run_pcmci, save_outputs
from src.pcmci.dependence import build_dependence_test
from src.pcmci.cmi import FAISSCMI

CONFIG_PATH = ROOT / 'config/corn/pcmci/cmi/climate_raw_monthly.json'

config = load_config(CONFIG_PATH)
print('Loaded config:', CONFIG_PATH)
print('Config name:', config.name)
print('Dependence method:', config.dependence.method)
print('Dependence params:', config.dependence.params)
print('PCMCI settings:', config.pcmci)


Loaded config: /Users/arshia/Projects/Papers/FinancialAI/Gaia/config/corn/pcmci/cmi/climate_raw_monthly.json
Config name: corn_climate_raw_monthly_cmi
Dependence method: cmi
Dependence params: {'k': 5, 'significance': 'shuffle_test', 'sig_samples': 200, 'use_gpu': True}
PCMCI settings: PCMCIConfig(tau_min=1, tau_max=5, pc_alpha=0.2, alpha_level=0.05, fdr_method='fdr_bh', max_conds_dim=None, max_combinations=1, max_conds_py=None, max_conds_px=None)


## Optional: shrink the run for a quick smoke test

Set `SMOKE_TEST = True` to make the notebook finish faster while still exercising the same code path.


In [4]:
SMOKE_TEST = True
SAVE_OUTPUTS = False

if SMOKE_TEST:
    config.name = f"{config.name}_smoke"
    config.pcmci.tau_max = 2
    config.pcmci.max_conds_dim = 1
    config.dependence.params['sig_samples'] = 25
    config.output.save_tigramite_plots = False
    config.output.save_networkx_plot = False
    config.output.max_links = 5

print('Effective config name:', config.name)
print('Effective dependence params:', config.dependence.params)
print('Effective tau_max:', config.pcmci.tau_max)
print('Effective max_conds_dim:', config.pcmci.max_conds_dim)


Effective config name: corn_climate_raw_monthly_cmi_smoke
Effective dependence params: {'k': 5, 'significance': 'shuffle_test', 'sig_samples': 25, 'use_gpu': True}
Effective tau_max: 2
Effective max_conds_dim: 1


In [5]:
dependence_test = build_dependence_test(config.dependence)
print('Dependence test type:', type(dependence_test).__name__)

if isinstance(dependence_test, FAISSCMI):
    print('FAISSCMI.use_gpu:', dependence_test.use_gpu)
    print('FAISSCMI.gpu_device:', dependence_test.gpu_device)
    print('FAISSCMI.res is not None:', dependence_test.res is not None)
    probe_index = dependence_test._build_index(2)
    print('FAISSCMI probe index type:', type(probe_index).__name__)
    print('FAISSCMI probe backend:', 'gpu' if 'gpu' in type(probe_index).__name__.lower() else 'cpu')
else:
    print('Loaded dependence test is not FAISSCMI.')


2026-05-15 19:11:36,062 INFO src.pcmci.cmi: FAISSCMI runtime: selected_backend=cpu, use_gpu=True, gpu_device=0, gpu_resources=False, faiss_gpu_count=0, torch_cuda_available=False, torch_mps_available=True, standardize=True, k=5, sig_samples=25
2026-05-15 19:11:36,064 INFO src.pcmci.cmi: FAISSCMI detected MPS support, but FAISS does not use MPS; the backend remains cpu.
2026-05-15 19:11:36,064 INFO src.pcmci.cmi: FAISSCMI requested GPU but FAISS GPU resources are unavailable; falling back to CPU indexes.
2026-05-15 19:11:36,065 INFO src.pcmci.cmi: FAISSCMI search backend: backend=cpu, index_type=IndexFlatL2


Dependence test type: FAISSCMI
FAISSCMI.use_gpu: True
FAISSCMI.gpu_device: 0
FAISSCMI.res is not None: False
FAISSCMI probe index type: IndexFlatL2
FAISSCMI probe backend: cpu


## Run PCMCI

This calls the same `run_pcmci(...)` function used by the script. The logging added in the project should print the selected backend.


In [6]:
payload = run_pcmci(config)

print('Run completed.')
print('Rows used:', payload['run_result'].row_count)
print('Selected columns:', len(payload['run_result'].selected_columns))
print('Significant links found:', len(payload['run_result'].links))
payload['run_result'].links[:5]


2026-05-15 19:11:38,258 INFO src.pcmci.runner: Starting PCMCI run: corn_climate_raw_monthly_cmi_smoke
2026-05-15 19:11:38,274 INFO src.pcmci.runner: Numeric frame ready: 192 rows x 6 columns — ['prcp', 'awnd', 'tmin', 'tmax', 'co2', 'pdsi']
2026-05-15 19:11:38,275 INFO src.pcmci.cmi: FAISSCMI runtime: selected_backend=cpu, use_gpu=True, gpu_device=0, gpu_resources=False, faiss_gpu_count=0, torch_cuda_available=False, torch_mps_available=True, standardize=True, k=5, sig_samples=25
2026-05-15 19:11:38,275 INFO src.pcmci.cmi: FAISSCMI detected MPS support, but FAISS does not use MPS; the backend remains cpu.
2026-05-15 19:11:38,275 INFO src.pcmci.cmi: FAISSCMI requested GPU but FAISS GPU resources are unavailable; falling back to CPU indexes.
2026-05-15 19:11:38,276 INFO src.pcmci.runner: Running PCMCI with CMI on 6 variables (tau_min=1, tau_max=2, pc_alpha=0.2, alpha_level=0.05)
2026-05-15 19:11:38,277 INFO src.pcmci.cmi: FAISSCMI search backend: backend=cpu, index_type=IndexFlatL2
2026-

Run completed.
Rows used: 192
Selected columns: 6
Significant links found: 0


[]

In [7]:
if SAVE_OUTPUTS:
    paths = save_outputs(
        config,
        payload['pcmci'],
        payload['run_result'],
        payload['results'],
    )
    print('Saved outputs:')
    for key, value in paths.items():
        print(f'  {key}: {value}')
else:
    print('SAVE_OUTPUTS is False; no files were written by this notebook cell.')


SAVE_OUTPUTS is False; no files were written by this notebook cell.


## How to interpret the result

For CMI, the strongest indicators are:
- `FAISSCMI runtime: selected_backend=...`
- `FAISSCMI probe backend: ...`
- `FAISSCMI search backend: backend=...`

Interpretation:
- `selected_backend=cuda` and `search backend: gpu` => CMI is using FAISS GPU.
- `selected_backend=cpu` and `search backend: cpu` => CMI is not using GPU.
- `torch.backends.mps.is_available() = True` alone does **not** mean CMI is on GPU; FAISS must expose GPU resources for that.
